In [1]:
import numpy as np
import pandas as pd
import json, os, random, pickle
from copy import deepcopy
from types import SimpleNamespace

import torch
import torch.nn.functional as F
from torch.nn import Linear, Dropout, BatchNorm1d, Sequential, ReLU
from torch_geometric.nn import GCNConv, GINEConv, GATv2Conv, global_mean_pool
from torch_geometric.loader import DataLoader
from torch.optim import Adam

from scipy.special import expit
from scipy.optimize import minimize_scalar

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    average_precision_score, precision_score,
    recall_score, classification_report, roc_curve
)
from xgboost import XGBClassifier

import sys
from pathlib import Path
sys.path.append(str(Path(PATH TO PBSG)))

import pbsg
from pbsg.polymer import polymers_from_df
from pbsg.polymer.generators import smiles_from_graphs
from pbsg.ml import to_pyg
from pbsg.ml.pyg import architecture_onehot
from pbsg.polymer.parser import parse_smiles_list
import ast

def safe_parse(val):
    try:
        if pd.isna(val): return None
    except (TypeError, ValueError): pass
    while isinstance(val, str):
        try: val = ast.literal_eval(val)
        except (ValueError, SyntaxError): break
    return val

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

/Users/xgv7306/miniconda3/envs/pbsg/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('data_classifier.csv')
for col in ['beads_list', 'lengths', 'reactivity_ratios', 'spatiality_rules']:
    if col in df.columns:
        df[col] = df[col].apply(safe_parse)
df['lengths']             = df['lengths'].apply(lambda x: None if isinstance(x, str) else x)
df['architecture_onehot'] = df.apply(architecture_onehot, axis=1)
df['beads_list']          = df['repeat_units'].apply(parse_smiles_list)

In [3]:
with open("random_stratified_splits_cls.json") as f:
    splits = json.load(f)

folds        = splits["folds"]
test_idx     = splits["test"]
all_idx      = np.arange(len(df))
trainval_idx = np.setdiff1d(all_idx, test_idx)
print(f"trainval={len(trainval_idx)}  test={len(test_idx)}  folds={len(folds)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

trainval=546  test=97  folds=5
Device: cpu


In [4]:
X_fp_RU_pooled = np.load("Vectors/X_fp_pooled_RU.npy")
X_fp_poly_pooled      = np.load("Vectors/X_fp_pooled_poly.npy")

y=np.load("Vectors/y.npy")

In [5]:
FP_NAMES     = [f"morgan_fp_{i}" for i in range(1024)]
DESC_NAMES   = ["MolWt","LogP","TPSA","RotBonds","Rings","HBD","HBA","Heteroatoms","NHOH",
                "AliphaticRings","AmideBonds","AromaticRings","backbone_atoms","sidechain_atoms",
                "avg_branch_length","flexibility","side_bulkiness_frac","kier_flex"]
ARCH_NAMES   = ["arch_alternating","arch_block","arch_random","arch_homopolymer"]
STEREO_NAMES = ["stereo_isotactic","stereo_syndiotactic","stereo_stereo-irregular",
                "stereo_atactic","stereo_achiral","stereo_unknown"]
SEQ_NAMES    = ["Pm","Px","asym"]
EDGE_NAMES   = ["edge_stereo"]
ALL_NAMES    = FP_NAMES + DESC_NAMES + ARCH_NAMES + STEREO_NAMES + SEQ_NAMES + EDGE_NAMES

print("total names:", len(ALL_NAMES))
assert len(ALL_NAMES) == 1056
assert X_fp_RU_pooled.shape[1] == len(ALL_NAMES)
print("names align with matrix width ✓")

total names: 1056
names align with matrix width ✓


In [6]:
STEREO_COLS = set(STEREO_NAMES + ["Pm"] + EDGE_NAMES)          # 6 + 1 + 1 = 8
keep_idx = [i for i, n in enumerate(ALL_NAMES) if n not in STEREO_COLS]
drop_idx = [i for i, n in enumerate(ALL_NAMES) if n in STEREO_COLS]

assert len(keep_idx) == 1048 and len(drop_idx) == 8, (len(keep_idx), len(drop_idx))
print("DROPPING:", [ALL_NAMES[i] for i in drop_idx])
print("kept SEQ (should include Px, asym):",
      [ALL_NAMES[i] for i in keep_idx if ALL_NAMES[i] in SEQ_NAMES])

DROPPING: ['stereo_isotactic', 'stereo_syndiotactic', 'stereo_stereo-irregular', 'stereo_atactic', 'stereo_achiral', 'stereo_unknown', 'Pm', 'edge_stereo']
kept SEQ (should include Px, asym): ['Px', 'asym']


In [7]:
def strip_stereo_pooled(X):
    assert X.shape[1] == 1056, X.shape
    return X[:, keep_idx]

X_ru_ab   = strip_stereo_pooled(X_fp_RU_pooled)      # (407, 1048)
X_poly_ab = strip_stereo_pooled(X_fp_poly_pooled)    # (407, 1048)

print("ablated RU  :", X_ru_ab.shape)
print("ablated poly:", X_poly_ab.shape)
# confirm it's genuinely different from full (not accidentally a view/no-op)
print("differs from full:", X_ru_ab.shape[1] != X_fp_RU_pooled.shape[1])

ablated RU  : (643, 1048)
ablated poly: (643, 1048)
differs from full: True


In [9]:
import importlib
import scripts.test_eval_ablation_models_cls
importlib.reload(scripts.test_eval_ablation_models_cls)

<module 'scripts.test_eval_ablation_models_cls' from '/Users/xgv7306/Library/CloudStorage/OneDrive-NorthwesternUniversity/northwestern/2-PolyID:Dora/polymer_melting_temp_ML/Section_S4/Classifier/scripts/test_eval_ablation_models_cls.py'>

In [11]:
from scripts.test_eval_ablation_models_cls import run_all_test

test_results = run_all_test(
    X_ru_ab, X_poly_ab,
    y, df, trainval_idx, test_idx, device,
    hp_path="best_hp_full.json",
)

Loaded HPs from best_hp_full.json  (21 entries)

── XGB (FP RU) ──────────────────────────────────────────

  TEST RESULTS: XGB (FP RU)
  Accuracy              : 0.8557
  Macro F1              : 0.8454
  AUROC                 : 0.8984
  AUPRC                 : 0.9349
  No Tm Precision       : 0.8286
  No Tm Recall          : 0.7838
  Has Tm Precision      : 0.8710
  Has Tm Recall         : 0.9000
  train_time_min        : 0.0100

  Subgroup: copolymer_type
  Group                    n      Acc       F1
  ---------------------------------------------
  alternating              9   0.5556   0.6667
  block                    2   1.0000   1.0000
  random                  30   0.8667   0.9259

  Subgroup: stereo_class
  Group                    n      Acc       F1
  ---------------------------------------------
  achiral                  5   1.0000   1.0000
  atactic                 19   0.7895   0.7500
  isotactic               45   0.8444   0.9014
  stereo-irregular         9   1.0000   1